In [3]:
#part1 = Importing
import os
import regex as re #This imports the regular expression
import pickle #used to save and load python objects , saving trained models, tokenizers , processed data
import time

from collections import Counter, defaultdict #count frequencies and provides defaukt values for missing keys
from datasets import load_dataset #imports the hugging face Datsets library
from tqdm.auto import tqdm #creates progress bars for loops.
print("All Imports are successful")

c:\Users\saniy\LLM_Tokenization\.venv-2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All Imports are successful


In [6]:
#Part-2 = GPT-2 style byte-to-unicode mapping
def bytes_to_unicode():
    """ Maps 256 bytes to 256 printables unicode chars."""
    bs = list(range(ord("!") , ord("~") +1)) + \
         list(range(ord("¡") , ord("¬") +1) ) + \
         list(range(ord("®") , ord("ÿ") +1)) 

    cs = bs[:] #creates the copy of bs
    n=0

    for b in range(256):
        if b not in bs:
            bs.append(b)  #append the b value in bs
            cs.append(256 + n) #assign new unicode numbers
            n+=1

    cs= [chr(c) for c in cs] #unicode the 65->A storing the values in  a dict 
    return dict(zip(bs,cs))


B2U = bytes_to_unicode()  #This function will return dictionary 
U2B ={v: k for k, v in B2U.items()} # reverse mapping (Decode if we want our original char)

def text_to_bytes(text): #converting the text into bytes 
    """ Convert text-> tuple of byte-level unicode chars."""
    return tuple(B2U[b] for b in text.encode('utf-8'))  #encode tuple([ B2U[72], B2U[101], B2U[108],B2U[108],B2U[111]]) 
#Mapping 

In [7]:
#Part3 = Pre- Tokenization 
PATTERN = re.compile( #d+ rule yaad rakhta (converts a regex pattern into a reusable regex)
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)


def pretokenize(text):
    """Split text into words using GPT-2 regex."""
    return PATTERN.findall(text)  #

In [8]:
#Part4= Data Loading

def load_tinystories(num_samples = 50000): #it loads the 50000 stories
    """ Load TinyStories dataset from HuggingFace."""
    print("Downloading TinyStories.....")
    dataset = load_dataset("roneneldan/TinyStories" , split = "train") #Dataset on the hugging face train using to get the trained data
    texts = dataset.select(range(num_samples))["text"] #returns the list of tthe stories in text 
    return "\n".join(texts) #Convert allstories in big string 


def load_wikitext():
    """ Load WikiText-103 from HuggingFace."""
    print("Downloading WikiText-103....")
    dataset = load_dataset("wikitext" , "wikitext-103-raw-v1", split = "train")
    return "\n".join(dataset["text"])


def load_custom_corpus(folder_path):
    """ Load all .txt files from a folder."""
    all_text = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith('.txt'):
            path = os.path.join(folder_path, filename)
            with open(path, 'r' , encoding = 'utf-8') as f:
                all_text.append(f.read())
                print(f"Loaded: {filename}")
    return "\n".join(all_text)

In [9]:
#part-5 = Cleaning 
def clean_text(text):
    """Basic text cleaning."""
    #syntax = re.sub(pattern, replacement, text)
    text = re.sub(r'\n{3,}', '\n\n', text)   # collapse "newlines" if there is more than 3 
    text = re.sub(r' {2,}', ' ', text)        # collapse "spaces"
    text = text.replace('\x00', '')           # remove null bytes
    return text.strip() #strip: Beginning aur end ke extra spaces/newlines hata deta hai.


In [10]:
#Part-6 Build Word Frequencies

def build_word_freqs(text):
    """ Returns: Counter mapping byte-tuple-> frequency
    example: {(b'T', b'h' , b'e') : 125430, .....}"""
    print("Pre-tokenizing.....")
    words = pretokenize(text) 
    print(f"Total pre-tokens : {len(words):,}") # count the length of pretokens

    print("Converting to bytes....") 
    word_freqs = Counter() #Creates the Empty frequency dictionary 
    for word in words: #The loop in every pretoken 
        byte_word = text_to_bytes(word)
        word_freqs[byte_word] += 1 #Checks how many pretokens are repeating 

    print(f"  Unique byte-words: {len(word_freqs):,}") #To get the total keys present in the dictionary Unique 
    return word_freqs

In [11]:
# ============================================================
# PROFILING: DATA PREPARATION
# ============================================================

print("=" * 60)
print("PROFILING DATA PIPELINE")
print("=" * 60)

# -------------------------
# 1. DATA LOADING
# -------------------------

start = time.perf_counter()

full_text = load_tinystories(num_samples=50000)

load_time = time.perf_counter() - start

print(f"Loaded: {len(full_text):,} chars")
print(f"⏱ Loading time: {load_time:.3f} sec")


# -------------------------
# 2. CLEANING
# -------------------------

start = time.perf_counter()

full_text = clean_text(full_text)

clean_time = time.perf_counter() - start

print(f"Cleaned: {len(full_text):,} chars")
print(f"⏱ Cleaning time: {clean_time:.3f} sec")


# -------------------------
# 3. PRETOKENIZATION
# -------------------------

start = time.perf_counter()

words = pretokenize(full_text)

pretokenize_time = time.perf_counter() - start

print(f"Total pre-tokens: {len(words):,}")
print(f"⏱ Pre-tokenization time: {pretokenize_time:.3f} sec")


# -------------------------
# 4. BYTE CONVERSION + FREQUENCY
# -------------------------

start = time.perf_counter()

word_freqs = Counter()

for word in words:
    byte_word = text_to_bytes(word)
    word_freqs[byte_word] += 1

frequency_time = time.perf_counter() - start

print(f"Unique byte-words: {len(word_freqs):,}")
print(f"⏱ Byte conversion + frequency time: {frequency_time:.3f} sec")


# -------------------------
# TOTAL
# -------------------------

data_pipeline_time = (
    load_time
    + clean_time
    + pretokenize_time
    + frequency_time
)

print("\n" + "=" * 60)
print("DATA PIPELINE PROFILE")
print("=" * 60)

print(f"Loading:                     {load_time:.3f} sec")
print(f"Cleaning:                    {clean_time:.3f} sec")
print(f"Pre-tokenization:            {pretokenize_time:.3f} sec")
print(f"Byte + frequency building:   {frequency_time:.3f} sec")
print(f"TOTAL:                       {data_pipeline_time:.3f} sec")

PROFILING DATA PIPELINE


Loaded: 44,607,052 chars
⏱ Loading time: 5.537 sec
Cleaned: 44,606,621 chars
⏱ Cleaning time: 0.640 sec
Total pre-tokens: 10,813,692
⏱ Pre-tokenization time: 4.316 sec
Unique byte-words: 20,450
⏱ Byte conversion + frequency time: 9.867 sec

DATA PIPELINE PROFILE
Loading:                     5.537 sec
Cleaning:                    0.640 sec
Pre-tokenization:            4.316 sec
Byte + frequency building:   9.867 sec
TOTAL:                       20.360 sec


In [12]:
os.makedirs("artifacts", exist_ok=True)

temp_path = "artifacts/word_freqs.pkl.tmp"
final_path = "artifacts/word_freqs.pkl"

with open(temp_path, "wb") as f:
    pickle.dump(word_freqs, f)
    f.flush()
    os.fsync(f.fileno())

os.replace(temp_path, final_path)

print(f"✅ Saved: {final_path}")
print(f"   Size: {os.path.getsize(final_path) / 1e6:.1f} MB")

✅ Saved: artifacts/word_freqs.pkl
   Size: 0.4 MB


In [13]:
def merge_word(word, pair, new_token):
    """Replace all non-overlapping occurrences of pair with new_token."""

    new_word = []
    i = 0

    while i < len(word):

        if (
            i < len(word) - 1
            and (word[i], word[i + 1]) == pair
        ):
            new_word.append(new_token)
            i += 2

        else:
            new_word.append(word[i])
            i += 1

    return tuple(new_word)

In [14]:
def get_word_pairs(word): #changed
    for i in range(len(word) - 1):
        yield (word[i], word[i + 1])


def add_word_to_pair_index(word, pair_to_words):
    for pair in get_word_pairs(word):
        pair_to_words[pair].add(word)


def remove_word_from_pair_index(word, pair_to_words):
    for pair in get_word_pairs(word):
        words = pair_to_words.get(pair)

        if words is not None:
            words.discard(word)

            if not words:
                del pair_to_words[pair]


def add_word_pairs_to_counts(word, freq, pair_counts):
    for pair in get_word_pairs(word):
        pair_counts[pair] += freq


def remove_word_pairs_from_counts(word, freq, pair_counts):
    for pair in get_word_pairs(word):
        pair_counts[pair] -= freq

        if pair_counts[pair] <= 0:
            del pair_counts[pair]

In [15]:
def train_bpe_v2(word_freqs, vocab_size, verbose=True):
    """
    V2 BPE training.

    Improvements:
    - Pair counts are calculated only once initially.
    - Only words containing the selected pair are updated.
    - Pair counts are updated incrementally.
    - pair_to_words tells us which words are affected.
    """

    # --------------------------------------------------
    # 1. Initial vocabulary
    # --------------------------------------------------

    vocab = set(B2U.values())

    print(f"Base vocab size:    {len(vocab)}")
    print(f"Target vocab size:  {vocab_size}")
    print(f"Merges to perform:  {vocab_size - len(vocab):,}")

    num_merges = vocab_size - len(vocab)

    # --------------------------------------------------
    # 2. Calculate pair counts ONCE
    # --------------------------------------------------

    print("\nBuilding initial pair counts...")

    pair_counts = Counter()

    for word, freq in word_freqs.items():
        add_word_pairs_to_counts(
            word,
            freq,
            pair_counts
        )

    print(f"Initial unique pairs: {len(pair_counts):,}")

    # --------------------------------------------------
    # 3. Build pair -> words index
    # --------------------------------------------------

    print("Building pair index...")

    pair_to_words = defaultdict(set)

    for word in word_freqs:
        add_word_to_pair_index(
            word,
            pair_to_words
        )

    print("Pair index ready.")

    # --------------------------------------------------
    # 4. BPE merge loop
    # --------------------------------------------------

    merges = {}

    pbar = tqdm(
        range(num_merges),
        desc="BPE V2 merges"
    )

    for i in pbar:

        # ----------------------------------------------
        # Find most frequent pair
        # ----------------------------------------------

        if not pair_counts:
            print("\nNo more pairs to merge.")
            break

        best_pair = max(
            pair_counts,
            key=pair_counts.get
        )

        best_freq = pair_counts[best_pair]

        # ----------------------------------------------
        # Create new token
        # ----------------------------------------------

        new_token = best_pair[0] + best_pair[1]

        merges[best_pair] = new_token
        vocab.add(new_token)

        # ----------------------------------------------
        # Find ONLY affected words
        # ----------------------------------------------

        affected_words = list(
            pair_to_words.get(best_pair, ())
        )

        # ----------------------------------------------
        # Transform affected words
        # ----------------------------------------------

        new_words = {}

        for old_word in affected_words:

            freq = word_freqs[old_word]

            # Remove old word from pair index
            remove_word_from_pair_index(
                old_word,
                pair_to_words
            )

            # Remove old pair contributions
            remove_word_pairs_from_counts(
                old_word,
                freq,
                pair_counts
            )

            # Perform the BPE merge
            new_word = merge_word(
                old_word,
                best_pair,
                new_token
            )

            # Combine frequencies if the new word
            # already exists
            new_words[new_word] = (
                new_words.get(new_word, 0) + freq
            )

            # Remove old word
            del word_freqs[old_word]

        # ----------------------------------------------
        # Add transformed words
        # ----------------------------------------------

        for new_word, freq in new_words.items():

            word_freqs[new_word] = (
                word_freqs.get(new_word, 0) + freq
            )

            # Add new pair contributions
            add_word_pairs_to_counts(
                new_word,
                freq,
                pair_counts
            )

            # Add new word to pair index
            add_word_to_pair_index(
                new_word,
                pair_to_words
            )

        # ----------------------------------------------
        # Progress display
        # ----------------------------------------------

        if verbose and (i + 1) % 100 == 0:

            r = ''.join(
                best_pair
            ).replace('Ġ', '␣')

            pbar.set_postfix({
                "last": r,
                "freq": best_freq,
                "affected": len(affected_words),
                "vocab": len(vocab)
            })

    return merges, vocab

In [16]:
def encode(text, merges):
    """
    Encode text → list of token strings using learned merges.
    Applies merges in the exact order they were learned.
    """
    all_tokens = []
    
    for word in pretokenize(text):
        byte_word = text_to_bytes(word)
        tokens = list(byte_word)
        
        # Apply each merge rule in order or you can simply call the function merge_word to merge
        for pair, new_token in merges.items():
            i = 0
            while i < len(tokens) - 1:
                if (tokens[i], tokens[i + 1]) == pair:
                    tokens[i:i + 2] = [new_token]
                else:
                    i += 1
    
        all_tokens.extend(tokens)
    
    return all_tokens

print("✅ encode function fixed")

✅ encode function fixed


In [17]:
#VOCAB -> ID Mapping 
def build_vocab_map(vocab):
    """ Assign integer IDs to every token.
    """
    return {token: idx for idx, token in enumerate(sorted(vocab))} #sorting (vocab)

def encode_to_ids(text, merges, vocab_map):
    """Encode text → list of integer token IDs."""
    tokens = encode(text, merges)
    ids = []
    for t in tokens:
        if t in vocab_map:
            ids.append(vocab_map[t])
        else:
            # Fallback: encode byte-by-byte
            for ch in t:
                ids.append(vocab_map[ch])
    return ids



In [18]:
#DECODING
def decode(token_ids , id_to_token):
    """Convert token IDs back to text."""
    byte_chars = ''.join(id_to_token[i] for i in token_ids) 
    raw_bytes = bytes(U2B[c] for c in byte_chars)
    return raw_bytes.decode('utf-8' , errors ='replace')

In [19]:
class BPETokenizer:
    """Complete BPE Tokenizer with train/encode/decode."""

    def __init__(self, merges, vocab_map):
        self.merges = merges
        self.vocab_map = vocab_map
        self.id_to_token = {v: k for k, v in vocab_map.items()}

    @classmethod
    def train(cls, word_freqs, vocab_size, verbose=True):
        """Train a new tokenizer from word frequencies."""

        merges, vocab = train_bpe_v2(
            word_freqs,
            vocab_size=vocab_size,
            verbose=verbose
        )

        vocab_map = build_vocab_map(vocab)

        return cls(merges, vocab_map)

    def encode(self, text):
        """Text → list of integer IDs."""
        return encode_to_ids(text, self.merges, self.vocab_map)

    def decode(self, ids):
        """Integer IDs → text."""
        return decode(ids, self.id_to_token)

    def tokenize(self, text):
        """Text → list of human-readable token strings."""
        return encode(text, self.merges)

    @property
    def vocab_size(self):
        return len(self.vocab_map)

    def save(self, path):
        with open(path, 'wb') as f:
            pickle.dump({
                'merges': self.merges,
                'vocab_map': self.vocab_map
            }, f)

        print(f"✅ Saved tokenizer: {path}")

    @classmethod
    def load(cls, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)

        print(f"✅ Loaded tokenizer: {path}")

        return cls(
            data['merges'],
            data['vocab_map']
        )


# Quick test
print("✅ BPETokenizer class ready")
print(
    "   Methods:",
    [m for m in dir(BPETokenizer) if not m.startswith('_')]
)

✅ BPETokenizer class ready
   Methods: ['decode', 'encode', 'load', 'save', 'tokenize', 'train', 'vocab_size']


In [20]:
import time
import pickle

# Fresh copy — important because V2 modifies word_freqs
with open("artifacts/word_freqs.pkl", "rb") as f:
    word_freqs_v2 = pickle.load(f)

start = time.perf_counter()

merges_v2, vocab_v2 = train_bpe_v2(
    word_freqs_v2,
    vocab_size=2000,
    verbose=True
)

v2_time = time.perf_counter() - start

print("\n" + "=" * 60)
print("V2 BENCHMARK")
print("=" * 60)

print(f"Training time : {v2_time:.3f} sec")
print(f"Merges        : {len(merges_v2):,}")
print(f"Vocab size    : {len(vocab_v2):,}")
print(f"Speedup       : {67.461 / v2_time:.2f}x")

Base vocab size:    256
Target vocab size:  2000
Merges to perform:  1,744

Building initial pair counts...
Initial unique pairs: 1,165
Building pair index...
Pair index ready.


BPE V2 merges: 100%|██████████| 1744/1744 [00:01<00:00, 1126.10it/s, last=bow, freq=1149, affected=7, vocab=1956]    


V2 BENCHMARK
Training time : 1.630 sec
Merges        : 1,744
Vocab size    : 2,000
Speedup       : 41.37x


In [21]:
vocab_map_v2 = build_vocab_map(vocab_v2)

tokenizer_v2 = BPETokenizer(
    merges_v2,
    vocab_map_v2
)

tokenizer_v2.save("artifacts/bpe_tokenizer.pkl")

✅ Saved tokenizer: artifacts/bpe_tokenizer.pkl


In [22]:
# TEST ENCODE / DECODE

tokenizer = BPETokenizer.load("artifacts/bpe_tokenizer.pkl")

test_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Hello world!",
    "Once upon a time, there was a little girl.",
]

print("=" * 60)
print("TOKENIZER TESTING")
print("=" * 60)

for text in test_sentences:
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)
    decoded = tokenizer.decode(ids)

    match = "PASS" if text == decoded else "FAIL"

    print(f"\n[{match}] Text:    {text!r}")
    print(f"      Tokens:  {tokens}")
    print(f"      IDs:     {ids}")
    print(f"      Decoded: {decoded!r}")

✅ Loaded tokenizer: artifacts/bpe_tokenizer.pkl
TOKENIZER TESTING

[PASS] Text:    'The quick brown fox jumps over the lazy dog.'
      Tokens:  ['The', 'Ġquick', 'Ġbrown', 'Ġfox', 'Ġjump', 's', 'Ġover', 'Ġthe', 'Ġla', 'zy', 'Ġdog', '.']
      IDs:     [108, 1616, 1019, 1256, 1385, 559, 1544, 1806, 1403, 650, 1148, 21]
      Decoded: 'The quick brown fox jumps over the lazy dog.'

[PASS] Text:    'Hello world!'
      Tokens:  ['Hello', 'Ġworld', '!']
      IDs:     [63, 1944, 0]
      Decoded: 'Hello world!'

[PASS] Text:    'Once upon a time, there was a little girl.'
      Tokens:  ['Once', 'Ġupon', 'Ġa', 'Ġtime', ',', 'Ġthere', 'Ġwas', 'Ġa', 'Ġlittle', 'Ġgirl', '.']
      IDs:     [92, 1872, 889, 1826, 18, 1810, 1898, 889, 1431, 1281, 21]
      Decoded: 'Once upon a time, there was a little girl.'


In [23]:
tokenizer = BPETokenizer.load("artifacts/bpe_tokenizer.pkl")

text = "Hello world!"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print("IDs:", ids)
print("Decoded:", decoded)
print("PASS:", text == decoded)

✅ Loaded tokenizer: artifacts/bpe_tokenizer.pkl
IDs: [63, 1944, 0]
Decoded: Hello world!
PASS: True


In [24]:
print("Notebook kernel working!")

Notebook kernel working!
